# CS462 Lab 7 — Notebook 1: Media Objects & UDFs

This notebook covers **Part 2 (Media UDFs)** and **Part 3 (UDF flavors)** of the lab.

You read a collection of **geotagged photos** directly from S3 as binary, then write
**User-Defined Functions (UDFs)** that crack each image open with Pillow to extract:

- **visual features** — pixel dimensions and mean brightness, and
- **EXIF metadata** — GPS latitude/longitude and the capture timestamp.

The `lat`, `lon`, and `capture_time` you pull out here are exactly the columns
**Notebook 2** feeds into Apache Sedona for spatial analysis. So feature extraction
is the step that turns an opaque binary blob into a row you can run spatial and
temporal queries over.

## Setup — a SparkSession that reads from S3 (given)

Same S3A / Requester-Pays configuration as Lab 6. Run this cell first. The Spark UI
will come up at <http://localhost:4040> once the session starts.

In [ ]:
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               DoubleType, TimestampType, StringType)

BUCKET = "s3a://torstengrabs-bc/lab7"

spark = (
    SparkSession.builder
    .appName("lab7-media-udfs")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.profile.ProfileCredentialsProvider")
    .config("spark.hadoop.fs.s3a.requester.pays.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .getOrCreate()
)
print("Spark", spark.version)

---
## Part 2 — Media objects with UDFs

### 2.1  Read the photos as binary (given)

Spark's `binaryFile` data source gives you one row per file, with the raw bytes in a
`content` column. Nothing is decoded yet — to Spark each photo is just an opaque blob.

In [ ]:
images = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .load(f"{BUCKET}/images/")
)
images.printSchema()
print("photo count:", images.count())
images.select("path", "length", "modificationTime").show(5, truncate=False)

### 2.2  Warm-up: a plain UDF for image dimensions (given, worked example)

A **UDF** wraps an ordinary Python function so Spark can apply it to a column. This one
opens the bytes with Pillow and returns a `(width, height)` struct. Study the shape —
your next two UDFs follow the same pattern: open `content`, compute something, return it.

In [ ]:
import io
from PIL import Image

@F.udf(returnType=StructType([
    StructField("width", IntegerType()),
    StructField("height", IntegerType()),
]))
def image_size(content):
    try:
        with Image.open(io.BytesIO(content)) as im:
            return (im.width, im.height)
    except Exception:
        return (None, None)

sized = images.withColumn("size", image_size("content"))
sized.select("path", "size.*").show(5, truncate=False)

### 2.3  Exercise — extract GPS + timestamp from EXIF (20 pts)

Each photo carries **EXIF** metadata, including where and when it was taken. Your job is
to write a UDF that returns the latitude, longitude, and capture time.

EXIF stores GPS coordinates in **degrees / minutes / seconds (DMS)** as a triple, plus a
hemisphere reference (`N`/`S` for latitude, `E`/`W` for longitude). You must convert that
to a single signed decimal-degree number with this formula:

$$\text{decimal} = \text{degrees} + \frac{\text{minutes}}{60} + \frac{\text{seconds}}{3600}$$

then **negate** the result when the reference is `S` (latitude) or `W` (longitude).

The EXIF tag layout you need is documented in the starter code. Fill in `dms_to_decimal`
(implement the formula) and the body of `extract_geo` (pull the tags, call your function,
parse the timestamp, and cope with photos that have no GPS tags at all).

In [ ]:
import io
from datetime import datetime
from PIL import Image

# ---------------------------------------------------------------------------
# EXIF layout you need (Pillow gives you these via Image.getexif()):
#
#   exif = im.getexif()
#   gps  = exif.get_ifd(0x8825)   # GPS sub-IFD: a dict keyed by tag id
#        gps[1] -> 'N' or 'S'   (latitude ref)    gps[2] -> (deg, min, sec) latitude
#        gps[3] -> 'E' or 'W'   (longitude ref)   gps[4] -> (deg, min, sec) longitude
#   sub  = exif.get_ifd(0x8769)   # Exif sub-IFD
#        sub[36867] -> 'YYYY:MM:DD HH:MM:SS'  (DateTimeOriginal)
#
# Some photos in this dataset have NO GPS tags on purpose — handle that.
# ---------------------------------------------------------------------------

def dms_to_decimal(dms, ref):
    """(deg, min, sec) triple + 'N'/'S'/'E'/'W' ref -> signed decimal degrees."""
    # TODO: implement
    #   decimal = degrees + minutes/60 + seconds/3600
    #   then negate if ref is 'S' or 'W'.
    raise NotImplementedError

@F.udf(returnType=StructType([
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("capture_time", TimestampType()),
]))
def extract_geo(content):
    try:
        with Image.open(io.BytesIO(content)) as im:
            exif = im.getexif()
            gps = exif.get_ifd(0x8825)
            sub = exif.get_ifd(0x8769)
            # TODO: read tags 1-4 from `gps`, convert with dms_to_decimal(),
            #       parse tag 36867 from `sub` into a datetime, and return
            #       (lat, lon, capture_time). Return (None, None, None) when
            #       the GPS tags are missing.
            raise NotImplementedError
    except Exception:
        return (None, None, None)

geo = sized.withColumn("geo", extract_geo("content"))
geo.select("path", "geo.*").show(5, truncate=False)

### 2.4  Exercise — mean brightness (10 pts)

Write a UDF `mean_brightness(content)` that returns the average pixel brightness of a
photo as a `double`. Hint: convert the image to grayscale with `im.convert("L")`, get
its `histogram()`, and compute the intensity-weighted mean
(`sum(i * count_i) / total_pixels`). Return `None` on failure.

In [ ]:
@F.udf(returnType=DoubleType())
def mean_brightness(content):
    try:
        with Image.open(io.BytesIO(content)) as im:
            # TODO: grayscale histogram -> intensity-weighted mean
            raise NotImplementedError
    except Exception:
        return None

geo.withColumn("brightness", mean_brightness("content")) \
   .select("path", "brightness").show(5, truncate=False)

### 2.5  Build the feature "frame" — and drop the blobs (given)

Now assemble the extracted columns into a tidy DataFrame and, crucially, **drop the
`content` column**. This is the key move from lecture: once you've extracted features,
project the binary blobs away so Spark never has to serialize and shuffle them through a
wide dependency. From here on the data is just small scalar columns — ordinary Spark.

We persist the result to a Parquet file that **Notebook 2 reads**. Note the saved frame
has no `content` column, so no image bytes ever hit disk here.

In [ ]:
features = (
    geo
    .withColumn("brightness", mean_brightness("content"))
    .select(
        "path",
        F.col("size.width").alias("width"),
        F.col("size.height").alias("height"),
        "brightness",
        F.col("geo.lat").alias("lat"),
        F.col("geo.lon").alias("lon"),
        F.col("geo.capture_time").alias("capture_time"),
    )
)
features.show(5, truncate=False)

# The physical plan below carries no `content` column past the projection:
features.explain(mode="formatted")

OUT = "file:///home/jovyan/work/photo_features.parquet"
features.write.mode("overwrite").parquet(OUT)
print("wrote feature frame (no blobs) ->", OUT)

### 2.6  Spark UI tour (5 pts)

With the cells above run, open the Spark UI at <http://localhost:4040> and save two
screenshots into a folder named `notebook1_screenshots/`:

- `jobs.png` — the **Jobs** tab, showing the jobs from this notebook.
- `sql.png` — the physical plan for the `features.write` parquet write (steps below).

**How to find the `features.write` physical plan:**

1. In the Spark UI top menu, click the **SQL / DataFrame** tab.
2. It lists every query Spark has run, oldest first. Find the row whose
   **Description** mentions `parquet` / `save` and whose **Submitted** time matches
   when you ran §2.5 — that's the write (it's the only one in this notebook).
3. Click that query's **Description** link to open its detail page.
4. The page draws the physical-plan DAG. Confirm your UDFs show up as
   **`BatchEvalPython`** nodes and that no `content` column is carried past the
   project. Click **Details** at the bottom to expand the full text plan if you'd
   like to read it. Screenshot this page and save it as `sql.png`.

> The same plan is printed inline by the `features.explain(mode="formatted")` call
> in §2.5 — the SQL/DataFrame tab is just the visual version Spark renders for the
> actual run.

---
## Part 3 — Flavors of UDFs

A **plain `@udf`** ships one row at a time across the JVM↔Python boundary, pickling each
value. A **`@pandas_udf`** receives a whole batch as a pandas `Series` (transferred via
Apache Arrow, no per-row pickling). Here you'll build the vectorized version of the
brightness UDF and time both.

> Note: image decoding is inherently per-image, so you'll still loop inside the pandas
> UDF. What the vectorized form removes is the per-row *serialization* overhead, not the
> per-image decode cost — something to address in your reflection.

### 3.1  Exercise — brightness as a pandas (vectorized) UDF (12 pts)

Reimplement mean brightness as a `pandas_udf`. It receives a `pd.Series` of `bytes` and
must return a `pd.Series` of `double`.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(DoubleType())
def mean_brightness_vec(content: pd.Series) -> pd.Series:
    # TODO: loop the batch, decode each image, compute mean brightness,
    #       and return a pd.Series of the same length.
    raise NotImplementedError

images.withColumn("b", mean_brightness_vec("content")) \
      .select("path", "b").show(5, truncate=False)

### 3.2  Time both UDFs (given)

Run a full `count()` over each so the whole dataset passes through the UDF, and compare.

In [ ]:
import time
def timed_count(df):
    t0 = time.time()
    n = df.count()
    return n, round(time.time() - t0, 1)

plain_n, plain_s = timed_count(images.withColumn("b", mean_brightness("content")).select("b"))
vec_n,   vec_s   = timed_count(images.withColumn("b", mean_brightness_vec("content")).select("b"))
print(f"plain  @udf : {plain_n} rows in {plain_s}s")
print(f"pandas_udf  : {vec_n} rows in {vec_s}s")

### 3.3  Write-up (8 pts)

You will probably find the two times are **close** (e.g., ~2.5s vs ~2.3s). That is the
expected result here, not a failure of the vectorized UDF — and explaining *why* is the
whole point of this question.

In the cell below, in 1–2 short paragraphs:

1. Report your two timings.
2. Explain *what* the `pandas_udf` removes (per-row pickling across the JVM↔Python
   boundary — it ships a whole Arrow batch instead) and *what it does not change*: both
   UDFs still call `Image.open` and decode every JPEG, and that decode is the dominant,
   CPU-bound cost. The vectorized form speeds up **data transport, not the compute**, so
   when compute dominates the overall gain is small.
3. Conclude with the takeaway: this does **not** mean vectorized UDFs aren't worth it.
   The win scales with how much of the runtime is serialization. Describe a workload
   where you *would* expect a large speed-up (many rows + cheap per-row work, e.g. a
   numeric transform over millions of rows, where pickling dominates), versus this one
   (few rows + an expensive per-row decode).

Then take a screenshot of the **SQL** tab for either timed query as
`notebook1_screenshots/udf_compare.png`.

In [ ]:
# Your write-up here (timings + why pandas_udf differs + what it does not speed up).